## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

##Package Installation





## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Installing the libraries with the specified version
!pip install unidecode==1.4.0 gensim==4.3.3 zeugma==0.41 fasttext==0.9.3 pandas==2.2.2 numpy==1.26.4 matplotlib==3.10.0 seaborn==0.13.2 nltk==3.9.1 scikit-learn==1.6.1 -q

In [ ]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
#!CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.5 -q

In [ ]:
!pip install --force-reinstall --no-deps numpy

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

## Importing necessary packages

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

### Downloading and Loading the model (Llama-2-13B-chat-GGUF)





In [ ]:
## Model configuration
model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"
model_basename = "llama-2-13b-chat.Q5_K_M.gguf"
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
    )

In [ ]:
# Define model parameters
lcpp_llm = Llama(
    model_path=model_path,
    n_threads=2,  # CPU cores
    n_batch=256,  # Should be between 1 and n_ctx, consider the amount of VRAM in your GPU.
    n_gpu_layers=32,  # Change this value based on your model and your GPU VRAM pool.
    n_ctx=4096,  # Context window
)

#### Define Response Function

In [ ]:
def response_from_model(prompt,max_tokens=128,temperature=0,top_p=0.95,top_k=50, repeat_penalty=1.2,stop=['INST'],echo=False):
    model_output = lcpp_llm(
      prompt=prompt,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k,
      repeat_penalty=repeat_penalty,
      stop=stop,
      echo=echo
    )

    return model_output['choices'][0]['text']

### Function to process response using llama

In [ ]:
# function to generate, process, and return the response from the LLM
def generate_llama_response(user_prompt,temperature=0.01):

    # System message
    system_message = """
    [INST]<<SYS>> Respond to the user question based on the user prompt<</SYS>>[/INST]
    """

    # Combine user_prompt and system_message to create the prompt
    prompt = f"{user_prompt}\n{system_message}"

    # Generate a response from the LLaMA model
    response = response_from_model(
        prompt=prompt,
        max_tokens=1024,
        temperature=temperature,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False
    )

    # Extract and return the response text
    response_text = response
    return response_text

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_prompt = "What is the protocol for managing sepsis in a critical care unit?"
response = generate_llama_response(user_prompt)
print(response)

* Lama model gave an insightful response, however the response is not in detail and formated. It is not also specific to the context document given.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_prompt = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response = generate_llama_response(user_prompt)
print(response)

* Lama model gave the common response, however the response is not specific to the context merk document given.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_prompt = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response = generate_llama_response(user_prompt)
print(response)

* Lama model gave the general response, however the response is not specific to the context merk document given.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_prompt = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response = generate_llama_response(user_prompt)
print(response)

* Lama model gave the general response, however the response is not specific to the context merk document given.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_prompt = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response = generate_llama_response(user_prompt)
print(response)

* Lama model gave the general treatment and precautions for leg injury, however the response is not specific to the context merk document given.

####***Observation***
* Llama generated responses with given user queries, however the responses are not specific or based on the context given in the merck medical manual.
* We will apply prompt engineering techniques further to refine the responses further.

## Question Answering using LLM with Prompt Engineering

### **Scenario 1**
        max_tokens=1024,
        **temperature**=0,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False

### Query 1: What is the protocol for managing sepsis in a critical care unit?

####  Applying Zero Shot Prompting

In [ ]:
#Applying Zero shot Prompting
user_prompt='''
          What is the protocol for managing sepsis in a critical care unit?

          Provide the answer in a structured format with headings in bold letters and bullet points

          Provide a comprehensive, step-by-step overview of the current  protocol for managing sepsis in an adult intensive care unit (ICU)
                '''
response = generate_llama_response(user_prompt,0)
print(response)

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

#### Applying Chain of thoughts

In [ ]:
user_prompt = ''' You are a board-certified general surgeon. Think step by step to answer the following:
(1) What are the most common signs and symptoms of acute appendicitis in adults?
(2) Is appendicitis typically curable with antibiotics alone? Under what circumstances might non-operative management be considered?
(3) If surgery is required, what is the standard surgical procedure, and what are its key approaches (e.g., laparoscopic or open)?
Support your answer with current clinical guidelines (e.g The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011).

  '''
response = generate_llama_response(user_prompt,0)
print(response)

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

#### Applying a hybrid approach chain of thought prompting and Role prompting

In [ ]:
user_prompt = ''' You are a board-certified dermatologist specializing in hair disorders.
Please provide a comprehensive, step-by-step analysis of sudden patchy hair loss in adults, structured as follows:

1. Most Likely Diagnosis & Differential Diagnoses
– List the primary condition associated with sudden, localized bald spots.
– Include 2–3 key alternative causes to consider (e.g., fungal infection, traction alopecia, etc.).

2. Common Underlying Causes or Triggers
– Mention potential triggers (e.g., stress, genetic predisposition, other autoimmune conditions).

3. Evidence-Based Treatment Options
– First-line therapies (e.g., intralesional corticosteroids).
– Second-line or emerging options (e.g., topical immunotherapy, JAK inhibitors).
– Note which treatments are suitable for limited vs. extensive disease.

4. Prognosis and When to Refer
– Typical course of the condition.
– Red flags that warrant specialist referral.

Base your response on current guidelines ( e.g The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011).

  '''
response = generate_llama_response(user_prompt,0)
print(response)

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

#### Applying hybrid prompting

In [ ]:
user_prompt = ''' You are a neurologist and rehabilitation medicine specialist with expertise in acquired brain injury.

Provide a comprehensive, step-by-step overview of evidence-based treatments for adults who have sustained a physical injury to brain tissue (e.g., traumatic brain injury or TBI) resulting in temporary or permanent impairment of brain function.
Organize your response under the following headings:

1. Acute-Phase Medical & Surgical Management
– Key interventions in the first hours to days (e.g., ICP monitoring, oxygenation, surgery).

2. Long-Term Therapies for Persistent Deficits
– Cognitive rehabilitation strategies.
– Management of behavioral, emotional, or psychiatric sequelae (e.g., depression, impulsivity).
– Pharmacologic and non-pharmacologic approaches.

3. Emerging and most Advanced Therapies
– Examples like neuromodulation (TMS, tDCS), virtual reality rehab, or pharmacologic neuroprotection (if evidence-supported).


Base your recommendations on current guidelines from  The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011.


  '''
response = generate_llama_response(user_prompt,0)
print(response)

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

#### Applying few shot prompting

In [ ]:
user_prompt = '''You are an AI assistant supporting clinical decision-making in wilderness and emergency settings.
For each query, provide a concise, structured, evidence-based response tailored to medical professionals (e.g., physicians, paramedics, wilderness medics).

Use clear headings, bullet points, and reference established guidelines (e.g., Wilderness Medical Society, ATLS, or ACEP).

Examples:

Q: A hiker sustains a deep laceration on the forearm with active bleeding 10 miles from trailhead. What are the key steps?
A:
1. Hemorrhage Control
– Direct pressure with clean dressing; tourniquet if life-threatening arterial bleed.

2. Wound Assessment & Irrigation
– Clean with potable or disinfected water (≥1 L if possible); remove visible debris.
– Do not close in wilderness setting (risk of infection).

3. Dressing & Evacuation
– Apply sterile dressing, splint if joint involved, evacuate within 24h for formal care.

(Per Wilderness Medical Society Guidelines, 2020)

Q: A climber falls 20 ft and complains of mid-back pain with leg numbness. How should you manage?
A:
1. Spinal Immobilization
– Assume spinal injury; log-roll, use improvised cervical collar, minimize movement.

2. Neurological Assessment
– Check motor/sensory function, bowel/bladder control (red flag for cord injury).

3. Urgent Evacuation
– Prioritize rapid extraction; avoid walking the patient. Use litter if available.

(Based on WMS Spinal Injury Guidelines, 2022)

—

Now answer the following by refering the publication (The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011):

Q: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?



  '''
response = generate_llama_response(user_prompt,0)
print(response)

####***Observation***
* With Prompt engineering the response became better than before
* With hybrid prompting Llama included examples in
* Responses are well structured with headings and bullets or numbers
* Llama referered merck medical manual as instructed in the prompt  

### **Scenario 2**
        max_tokens=1024,
        **temperature**=0.8,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False

### Query 1: What is the protocol for managing sepsis in a critical care unit?

#### Applying Zero shot prompting

In [ ]:
#Applying Zero shot Prompting
user_prompt='''
          What is the protocol for managing sepsis in a critical care unit?

          Provide the answer in a structured format with headings in bold letters and bullet points

          Provide a comprehensive, step-by-step overview of the current  protocol for managing sepsis in an adult intensive care unit (ICU)
                '''
response = generate_llama_response(user_prompt,0.8)
print(response)

* With Prompt engineering the response became better than before

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_prompt = '''You are a board-certified general surgeon.

Think step by step to answer the following:

(1) What are the most common signs and symptoms of acute appendicitis in adults?
(2) Is appendicitis typically curable with antibiotics alone? Under what circumstances might non-operative management be considered?
(3) If surgery is required, what is the standard surgical procedure, and what are its key approaches (e.g., laparoscopic or open)?

Support your answer with current clinical guidelines (e.g The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011).


  '''
response = generate_llama_response(user_prompt,0.8)
print(response)

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_prompt = ''' You are a board-certified dermatologist specializing in hair disorders.
Please provide a comprehensive, step-by-step analysis of sudden patchy hair loss in adults, structured as follows:


1. Most Likely Diagnosis & Differential Diagnoses
– List the primary condition associated with sudden, localized bald spots.
– Include 2–3 key alternative causes to consider (e.g., fungal infection, traction alopecia, etc.).


2. Common Underlying Causes or Triggers
– Mention potential triggers (e.g., stress, genetic predisposition, other autoimmune conditions).


3. Evidence-Based Treatment Options
– First-line therapies (e.g., intralesional corticosteroids).
– Second-line or emerging options (e.g., topical immunotherapy, JAK inhibitors).
– Note which treatments are suitable for limited vs. extensive disease.


4. Prognosis and When to Refer
– Typical course of the condition.
– Red flags that warrant specialist referral.


Base your response on current guidelines ( e.g The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011).



  '''
response = generate_llama_response(user_prompt,0.8)
print(response)

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_prompt = ''' You are a neurologist and rehabilitation medicine specialist with expertise in acquired brain injury.


Provide a comprehensive, step-by-step overview of evidence-based treatments for adults who have sustained a physical injury to brain tissue (e.g., traumatic brain injury or TBI) resulting in temporary or permanent impairment of brain function.
Organize your response under the following headings:


1. Acute-Phase Medical & Surgical Management
– Key interventions in the first hours to days (e.g., ICP monitoring, oxygenation, surgery).


2. Long-Term Therapies for Persistent Deficits
– Cognitive rehabilitation strategies.
– Management of behavioral, emotional, or psychiatric sequelae (e.g., depression, impulsivity).
– Pharmacologic and non-pharmacologic approaches.


3. Emerging and most Advanced Therapies
– Examples like neuromodulation (TMS, tDCS), virtual reality rehab, or pharmacologic neuroprotection (if evidence-supported).




Base your recommendations on current guidelines from  The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011.


  '''
response = generate_llama_response(user_prompt,0.8)
print(response)

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_prompt = ''' You are an AI assistant supporting clinical decision-making in wilderness and emergency settings.
For each query, provide a concise, structured, evidence-based response tailored to medical professionals (e.g., physicians, paramedics, wilderness medics).


Use clear headings, bullet points, and reference established guidelines (e.g., Wilderness Medical Society, ATLS, or ACEP).


Examples:


Q: A hiker sustains a deep laceration on the forearm with active bleeding 10 miles from trailhead. What are the key steps?
A:
1. Hemorrhage Control
– Direct pressure with clean dressing; tourniquet if life-threatening arterial bleed.


2. Wound Assessment & Irrigation
– Clean with potable or disinfected water (≥1 L if possible); remove visible debris.
– Do not close in wilderness setting (risk of infection).


3. Dressing & Evacuation
– Apply sterile dressing, splint if joint involved, evacuate within 24h for formal care.


(Per Wilderness Medical Society Guidelines, 2020)


Q: A climber falls 20 ft and complains of mid-back pain with leg numbness. How should you manage?
A:
1. Spinal Immobilization
– Assume spinal injury; log-roll, use improvised cervical collar, minimize movement.


2. Neurological Assessment
– Check motor/sensory function, bowel/bladder control (red flag for cord injury).


3. Urgent Evacuation
– Prioritize rapid extraction; avoid walking the patient. Use litter if available.


(Based on WMS Spinal Injury Guidelines, 2022)


—


Now answer the following by refering the publication (The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011):


Q: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?



  '''
response = generate_llama_response(user_prompt,0.8)
print(response)

####***Observation***
* The model responses  became a slight hallucinated
* With hybrid prompting Llama included examples in
* Responses are well structured with headings and bullets or numbers
* Llama referered merck medical manual as instructed in the prompt  

### **Scenario 3**
        max_tokens=1024,
        **temperature**=10,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
#Applying Zero shot Prompting
user_prompt='''
          What is the protocol for managing sepsis in a critical care unit?

          Provide the answer in a structured format with headings in bold letters and bullet points

          Provide a comprehensive, step-by-step overview of the current  protocol for managing sepsis in an adult intensive care unit (ICU)
                '''
response = generate_llama_response(user_prompt,10)
print(response)

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_prompt = ''' You are a board-certified general surgeon.
Think step by step to answer the following:


(1) What are the most common signs and symptoms of acute appendicitis in adults?
(2) Is appendicitis typically curable with antibiotics alone? Under what circumstances might non-operative management be considered?
(3) If surgery is required, what is the standard surgical procedure, and what are its key approaches (e.g., laparoscopic or open)?


Support your answer with current clinical guidelines (e.g The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011).

  '''
response = generate_llama_response(user_prompt,10)
print(response)

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_prompt = ''' You are a board-certified dermatologist specializing in hair disorders.
Please provide a comprehensive, step-by-step analysis of sudden patchy hair loss in adults, structured as follows:


1. Most Likely Diagnosis & Differential Diagnoses
– List the primary condition associated with sudden, localized bald spots.
– Include 2–3 key alternative causes to consider (e.g., fungal infection, traction alopecia, etc.).


2. Common Underlying Causes or Triggers
– Mention potential triggers (e.g., stress, genetic predisposition, other autoimmune conditions).


3. Evidence-Based Treatment Options
– First-line therapies (e.g., intralesional corticosteroids).
– Second-line or emerging options (e.g., topical immunotherapy, JAK inhibitors).
– Note which treatments are suitable for limited vs. extensive disease.


4. Prognosis and When to Refer
– Typical course of the condition.
– Red flags that warrant specialist referral.


Base your response on current guidelines ( e.g The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011).

  '''
response = generate_llama_response(user_prompt,10)
print(response)

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_prompt = ''' You are a neurologist and rehabilitation medicine specialist with expertise in acquired brain injury.


Provide a comprehensive, step-by-step overview of evidence-based treatments for adults who have sustained a physical injury to brain tissue (e.g., traumatic brain injury or TBI) resulting in temporary or permanent impairment of brain function.
Organize your response under the following headings:


1. Acute-Phase Medical & Surgical Management
– Key interventions in the first hours to days (e.g., ICP monitoring, oxygenation, surgery).


2. Long-Term Therapies for Persistent Deficits
– Cognitive rehabilitation strategies.
– Management of behavioral, emotional, or psychiatric sequelae (e.g., depression, impulsivity).
– Pharmacologic and non-pharmacologic approaches.


3. Emerging and most Advanced Therapies
– Examples like neuromodulation (TMS, tDCS), virtual reality rehab, or pharmacologic neuroprotection (if evidence-supported).




Base your recommendations on current guidelines from  The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011.





  '''
response = generate_llama_response(user_prompt,10)
print(response)

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_prompt = ''' You are an AI assistant supporting clinical decision-making in wilderness and emergency settings.
For each query, provide a concise, structured, evidence-based response tailored to medical professionals (e.g., physicians, paramedics, wilderness medics).


Use clear headings, bullet points, and reference established guidelines (e.g., Wilderness Medical Society, ATLS, or ACEP).


Examples:


Q: A hiker sustains a deep laceration on the forearm with active bleeding 10 miles from trailhead. What are the key steps?
A:
1. Hemorrhage Control
– Direct pressure with clean dressing; tourniquet if life-threatening arterial bleed.


2. Wound Assessment & Irrigation
– Clean with potable or disinfected water (≥1 L if possible); remove visible debris.
– Do not close in wilderness setting (risk of infection).


3. Dressing & Evacuation
– Apply sterile dressing, splint if joint involved, evacuate within 24h for formal care.


(Per Wilderness Medical Society Guidelines, 2020)


Q: A climber falls 20 ft and complains of mid-back pain with leg numbness. How should you manage?
A:
1. Spinal Immobilization
– Assume spinal injury; log-roll, use improvised cervical collar, minimize movement.


2. Neurological Assessment
– Check motor/sensory function, bowel/bladder control (red flag for cord injury).


3. Urgent Evacuation
– Prioritize rapid extraction; avoid walking the patient. Use litter if available.


(Based on WMS Spinal Injury Guidelines, 2022)


—


Now answer the following by refering the publication (The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011):


Q: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?



  '''
response = generate_llama_response(user_prompt,10)
print(response)

####***Observation***
* Model starts hallucinating, Question 3 response is a conversation between a patient and a doctor.
* The responses lack specificness to the question asked
* Responses are well structured with headings and bullets or numbers
* Llama referered merck medical manual as instructed in the prompt  

### **Scenario 4**
        max_tokens=1024,
        **temperature**=5,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
#Applying Zero shot Prompting
user_prompt='''
          What is the protocol for managing sepsis in a critical care unit?

          Provide the answer in a structured format with headings in bold letters and bullet points

          Provide a comprehensive, step-by-step overview of the current  protocol for managing sepsis in an adult intensive care unit (ICU)
                '''
response = generate_llama_response(user_prompt,5)
print(response)

* With Prompt engineering the response became better than before

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_prompt = ''' You are a board-certified general surgeon.
Think step by step to answer the following:


(1) What are the most common signs and symptoms of acute appendicitis in adults?
(2) Is appendicitis typically curable with antibiotics alone? Under what circumstances might non-operative management be considered?
(3) If surgery is required, what is the standard surgical procedure, and what are its key approaches (e.g., laparoscopic or open)?


Support your answer with current clinical guidelines (e.g The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011).


  '''
response = generate_llama_response(user_prompt,5)
print(response)

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_prompt = ''' You are a board-certified dermatologist specializing in hair disorders.
Please provide a comprehensive, step-by-step analysis of sudden patchy hair loss in adults, structured as follows:


1. Most Likely Diagnosis & Differential Diagnoses
– List the primary condition associated with sudden, localized bald spots.
– Include 2–3 key alternative causes to consider (e.g., fungal infection, traction alopecia, etc.).


2. Common Underlying Causes or Triggers
– Mention potential triggers (e.g., stress, genetic predisposition, other autoimmune conditions).


3. Evidence-Based Treatment Options
– First-line therapies (e.g., intralesional corticosteroids).
– Second-line or emerging options (e.g., topical immunotherapy, JAK inhibitors).
– Note which treatments are suitable for limited vs. extensive disease.


4. Prognosis and When to Refer
– Typical course of the condition.
– Red flags that warrant specialist referral.


Base your response on current guidelines ( e.g The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011).



  '''
response = generate_llama_response(user_prompt,5)
print(response)

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_prompt = ''' You are a neurologist and rehabilitation medicine specialist with expertise in acquired brain injury.


Provide a comprehensive, step-by-step overview of evidence-based treatments for adults who have sustained a physical injury to brain tissue (e.g., traumatic brain injury or TBI) resulting in temporary or permanent impairment of brain function.
Organize your response under the following headings:


1. Acute-Phase Medical & Surgical Management
– Key interventions in the first hours to days (e.g., ICP monitoring, oxygenation, surgery).


2. Long-Term Therapies for Persistent Deficits
– Cognitive rehabilitation strategies.
– Management of behavioral, emotional, or psychiatric sequelae (e.g., depression, impulsivity).
– Pharmacologic and non-pharmacologic approaches.


3. Emerging and most Advanced Therapies
– Examples like neuromodulation (TMS, tDCS), virtual reality rehab, or pharmacologic neuroprotection (if evidence-supported).




Base your recommendations on current guidelines from  The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011.



  '''
response = generate_llama_response(user_prompt,5)
print(response)

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_prompt = ''' You are an AI assistant supporting clinical decision-making in wilderness and emergency settings.
For each query, provide a concise, structured, evidence-based response tailored to medical professionals (e.g., physicians, paramedics, wilderness medics).


Use clear headings, bullet points, and reference established guidelines (e.g., Wilderness Medical Society, ATLS, or ACEP).


Examples:


Q: A hiker sustains a deep laceration on the forearm with active bleeding 10 miles from trailhead. What are the key steps?
A:
1. Hemorrhage Control
– Direct pressure with clean dressing; tourniquet if life-threatening arterial bleed.


2. Wound Assessment & Irrigation
– Clean with potable or disinfected water (≥1 L if possible); remove visible debris.
– Do not close in wilderness setting (risk of infection).


3. Dressing & Evacuation
– Apply sterile dressing, splint if joint involved, evacuate within 24h for formal care.


(Per Wilderness Medical Society Guidelines, 2020)


Q: A climber falls 20 ft and complains of mid-back pain with leg numbness. How should you manage?
A:
1. Spinal Immobilization
– Assume spinal injury; log-roll, use improvised cervical collar, minimize movement.


2. Neurological Assessment
– Check motor/sensory function, bowel/bladder control (red flag for cord injury).


3. Urgent Evacuation
– Prioritize rapid extraction; avoid walking the patient. Use litter if available.


(Based on WMS Spinal Injury Guidelines, 2022)


—


Now answer the following by refering the publication (The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011):


Q: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?


  '''
response = generate_llama_response(user_prompt,5)
print(response)

####***Observation***
* Model started hallucinating
* The responses lack specificness to the question asked
* Responses became less structured.
* Llama referered merck medical manual as instructed in the prompt  

### **Scenario 5**
        max_tokens=1024,
        **temperature**=15,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
#Applying Zero shot Prompting
user_prompt='''
          What is the protocol for managing sepsis in a critical care unit?

          Provide the answer in a structured format with headings in bold letters and bullet points

          Provide a comprehensive, step-by-step overview of the current  protocol for managing sepsis in an adult intensive care unit (ICU)
                '''
response = generate_llama_response(user_prompt,15)
print(response)

* With Prompt engineering the response became better than before

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_prompt = ''' You are a board-certified general surgeon.
Think step by step to answer the following:


(1) What are the most common signs and symptoms of acute appendicitis in adults?
(2) Is appendicitis typically curable with antibiotics alone? Under what circumstances might non-operative management be considered?
(3) If surgery is required, what is the standard surgical procedure, and what are its key approaches (e.g., laparoscopic or open)?


Support your answer with current clinical guidelines (e.g The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011).

  '''
response = generate_llama_response(user_prompt,15)
print(response)

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_prompt = ''' You are a board-certified dermatologist specializing in hair disorders.
Please provide a comprehensive, step-by-step analysis of sudden patchy hair loss in adults, structured as follows:


1. Most Likely Diagnosis & Differential Diagnoses
– List the primary condition associated with sudden, localized bald spots.
– Include 2–3 key alternative causes to consider (e.g., fungal infection, traction alopecia, etc.).


2. Common Underlying Causes or Triggers
– Mention potential triggers (e.g., stress, genetic predisposition, other autoimmune conditions).


3. Evidence-Based Treatment Options
– First-line therapies (e.g., intralesional corticosteroids).
– Second-line or emerging options (e.g., topical immunotherapy, JAK inhibitors).
– Note which treatments are suitable for limited vs. extensive disease.


4. Prognosis and When to Refer
– Typical course of the condition.
– Red flags that warrant specialist referral.


Base your response on current guidelines ( e.g The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011).



  '''
response = generate_llama_response(user_prompt,15)
print(response)

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_prompt = '''You are a neurologist and rehabilitation medicine specialist with expertise in acquired brain injury.


Provide a comprehensive, step-by-step overview of evidence-based treatments for adults who have sustained a physical injury to brain tissue (e.g., traumatic brain injury or TBI) resulting in temporary or permanent impairment of brain function.
Organize your response under the following headings:


1. Acute-Phase Medical & Surgical Management
– Key interventions in the first hours to days (e.g., ICP monitoring, oxygenation, surgery).


2. Long-Term Therapies for Persistent Deficits
– Cognitive rehabilitation strategies.
– Management of behavioral, emotional, or psychiatric sequelae (e.g., depression, impulsivity).
– Pharmacologic and non-pharmacologic approaches.


3. Emerging and most Advanced Therapies
– Examples like neuromodulation (TMS, tDCS), virtual reality rehab, or pharmacologic neuroprotection (if evidence-supported).




Base your recommendations on current guidelines from  The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011.



  '''
response = generate_llama_response(user_prompt,15)
print(response)

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_prompt = ''' You are an AI assistant supporting clinical decision-making in wilderness and emergency settings.
For each query, provide a concise, structured, evidence-based response tailored to medical professionals (e.g., physicians, paramedics, wilderness medics).


Use clear headings, bullet points, and reference established guidelines (e.g., Wilderness Medical Society, ATLS, or ACEP).


Examples:


Q: A hiker sustains a deep laceration on the forearm with active bleeding 10 miles from trailhead. What are the key steps?
A:
1. Hemorrhage Control
– Direct pressure with clean dressing; tourniquet if life-threatening arterial bleed.


2. Wound Assessment & Irrigation
– Clean with potable or disinfected water (≥1 L if possible); remove visible debris.
– Do not close in wilderness setting (risk of infection).


3. Dressing & Evacuation
– Apply sterile dressing, splint if joint involved, evacuate within 24h for formal care.


(Per Wilderness Medical Society Guidelines, 2020)


Q: A climber falls 20 ft and complains of mid-back pain with leg numbness. How should you manage?
A:
1. Spinal Immobilization
– Assume spinal injury; log-roll, use improvised cervical collar, minimize movement.


2. Neurological Assessment
– Check motor/sensory function, bowel/bladder control (red flag for cord injury).


3. Urgent Evacuation
– Prioritize rapid extraction; avoid walking the patient. Use litter if available.


(Based on WMS Spinal Injury Guidelines, 2022)


—


Now answer the following by refering the publication (The Merck Manual of Diagnosis & Therapy, 19th Edition published in 2011):


Q: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?



  '''
response = generate_llama_response(user_prompt,15)
print(response)

####***Observation***
* Model started hallucinating
* The responses lack specificness to the question asked
* Responses became less structured.


* **Prompt engineering definitely made the responses refined, organized and well**
* **By varying temperature values, model gave different responses for each questionare**.
* **Model became more creative as temperature increased**

## Data Preparation for RAG

### Loading the Data

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
pdf_file = "/content/drive/MyDrive/content/drive/medical_diagnosis_manual.pdf"

In [ ]:
pdf_loader = PyMuPDFLoader(pdf_file)

In [ ]:
merck_manual = pdf_loader.load()

### Data Overview

#### Checking the 15th page

In [ ]:
print(merck_manual[15].page_content)


#### Checking the number of pages

In [ ]:
len(merck_manual)

### Data Chunking
Slicing the data into chunks facilitates fine-grained control of the specific information that can be injected as context.

In [ ]:
# Define Chunking Strategies
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap= 20
)

In [ ]:
# Split into chunks using chosen chunking strategy
document_chunks = pdf_loader.load_and_split(text_splitter)

### Total Chunks

In [ ]:
# Total number of chunks
len(document_chunks)

* There are 8477 chunks

### Checking Overlaps

In [ ]:
document_chunks[100].page_content

In [ ]:
document_chunks[101].page_content

* As expected there is overlapping among content in chunks
* To increase overlapping increase chunk_overlap

### Embedding

- Embeddings are a type of word representation that allows words with similar meaning to have a similar representation.
- They capture semantic properties of words and relations with other words.


In [ ]:
# Embedding model
from langchain_community.embeddings import SentenceTransformerEmbeddings
embedding_model = SentenceTransformerEmbeddings(model_name="thenlper/gte-large")

In [ ]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [ ]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

* The embedding model provides a fixed-length vector for any number of chunks.  
* This is necessary because we want to compare them for similarity.

### Vector Database

In [ ]:
out_dir = 'merck_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [ ]:
# Build Vector Database from Chroma
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [ ]:
# Save VectorDB
vectorstore.persist()

In [ ]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)
#vectorstore.get()

In [ ]:
vectorstore.embeddings

In [ ]:
# Check Similarity
vectorstore.similarity_search("Eyes Glasses Vision Scenery ",k=3)

* Similarity Search retrieved some context information from the medical manual

### Retriever

In [ ]:
# Define retriever
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 2}
)

#### Test retriver whether it gives relevant responses

In [ ]:

user_prompt='''
          What is the protocol for managing sepsis in a critical care unit?


                '''
relevant_document_chunks = retriever.invoke(user_prompt)


In [ ]:
for document in relevant_document_chunks:
    print(document.page_content.replace("\t", " "))

#### ***Observation***
* The response clearly shows the contextual information from the document

### Define System and User Prompt Template for RAG

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [ ]:
# System message
qna_system_message = """
You are a medical assistant whose work is to review the report and provide the appropriate answers from the context.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Do not mention anything about the context in your final answer.

If the answer is not found in the context, respond "I don't know".
"""

In [ ]:
# User message
qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

### Response Function for RAG

In [ ]:
# Define tuned response function
def RAG(user_input , llm,temperature=0.4,top_k=25):
    """
    Args:
        user_input: Takes a user input for which the response should be retrieved from the vectorDB.
        llm: The LLM to be used for generating the response
    Returns:
        The generated response based on the user query and the context from the knowledge base
    """
    global qna_system_message,qna_user_message_template
    relevant_document_chunks = retriever.invoke(user_input)
    context_list = [d.page_content.replace("\t", " ") for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)



    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""


    # Quering an LLM
    try:
        response = llm(
                prompt=prompt,
                max_tokens=500,
                temperature=temperature,
                top_p=0.80,
                repeat_penalty=1.2,
                top_k=top_k,
                stop=['INST'],
                echo=False
                )

        prediction =  response["choices"][0]["text"]

    except Exception as e:
        prediction = f'Sorry, I encountered the following error: \n {e}'

    return  prediction

## Question Answering using RAG

* We set the llm parameters as follows for the following Scenarios,
                max_tokens=500,
                top_p=0.80,
                repeat_penalty=1.2,
                stop=['INST'],
                echo=False,
                top_k=25
*  To compare the responses consistently, we changed only temperature values in 6 scenarios

### **Scenario 1**
        **temperature**=0.4,
        top_k=25

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input='''
          What is the protocol for managing sepsis in a critical care unit?'''
print(RAG(user_input,lcpp_llm,0.4))

* The response of lamma became more specific when compared to without context
* Generated responses by referring merck medical manual given

In [ ]:
user_input='''
          What is the protocol for managing sepsis in a critical care unit?

          Provide the answer in a structured format with headings in bold letters and bullet points

          Provide a comprehensive, step-by-step overview of the current  protocol for managing sepsis in an adult intensive care unit (ICU)
                '''
print(RAG(user_input,lcpp_llm,0.4))

* The question with prompt engineering techniques provided more detailed answer.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(RAG(user_input,lcpp_llm,0.4))

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(RAG(user_input,lcpp_llm,0.4))

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(RAG(user_input,lcpp_llm,0.4))

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(RAG(user_input,lcpp_llm,0.4))

####***Observation***
* Model responses seems contextual
* With temperature=0.4, the model is less creative


### Fine-tuning
#### We will further fine tune the model by changing temperature values and try to find an optimum value by analyzing the results

               

### **Scenario 2**
        **temperature**=0,
        top_k=25

### Retrieve the context from the medical manual related to Question 1

In [ ]:
user_prompt='''What is the protocol for managing sepsis in a critical care unit?'''
relevant_document_chunks = retriever.invoke(user_prompt)


In [ ]:
for document in relevant_document_chunks:
    print(document.page_content.replace("\t", " "))

* The retrieved information has relevant information of the context referring to the given query

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input='''
          What is the protocol for managing sepsis in a critical care unit?'''
print(RAG(user_input,lcpp_llm,0,25))

* Got the response based on the context given

In [ ]:
user_input='''
          What is the protocol for managing sepsis in a critical care unit?

          Provide the answer in a structured format with headings in bold letters and bullet points

          Provide a comprehensive, step-by-step overview of the current  protocol for managing sepsis in an adult intensive care unit (ICU)
                '''
print(RAG(user_input,lcpp_llm,0,25))

* Applying Prompt Engineering provided a well structured response

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(RAG(user_input,lcpp_llm,0,25))

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(RAG(user_input,lcpp_llm,0,25))

* Got the response based on the given context

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(RAG(user_input,lcpp_llm,0,25))

* Got the response based on the context given

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(RAG(user_input,lcpp_llm,0,25))

####***Observation***
* Model responses became context specific
* There is no randomness in the response as temperature=0

### **Scenario 3**
        **temperature**=4,
        top_k=25

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input='''
          What is the protocol for managing sepsis in a critical care unit?
            '''
print(RAG(user_input,lcpp_llm,temperature=4,top_k=25))

In [ ]:
user_input='''
          What is the protocol for managing sepsis in a critical care unit?

          Provide the answer in a structured format with headings in bold letters and bullet points

          Provide a comprehensive, step-by-step overview of the current  protocol for managing sepsis in an adult intensive care unit (ICU)
                '''
print(RAG(user_input,lcpp_llm,temperature=4,top_k=25))

* Response is well structured and detail

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(RAG(user_input,lcpp_llm,temperature=4,top_k=25))

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(RAG(user_input,lcpp_llm,temperature=4,top_k=25))

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(RAG(user_input,lcpp_llm,temperature=4,top_k=25))

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(RAG(user_input,lcpp_llm,temperature=4,top_k=25))

####***Observation***
* Model responses are good, however less specific to context


### **Scenario 4**
        **temperature**=6,
        top_k=25

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input='''
          What is the protocol for managing sepsis in a critical care unit?
            '''
print(RAG(user_input,lcpp_llm,temperature=6,top_k=25))

In [ ]:
user_input='''
          What is the protocol for managing sepsis in a critical care unit?

          Provide the answer in a structured format with headings in bold letters and bullet points

          Provide a comprehensive, step-by-step overview of the current  protocol for managing sepsis in an adult intensive care unit (ICU)
                '''
print(RAG(user_input,lcpp_llm,temperature=6,top_k=25))

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(RAG(user_input,lcpp_llm,temperature=6,top_k=25))

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(RAG(user_input,lcpp_llm,temperature=6,top_k=25))

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(RAG(user_input,lcpp_llm,temperature=6,top_k=25))

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(RAG(user_input,lcpp_llm,temperature=6,top_k=25))

####***Observation***
* Model started hallucinating
* The response of Question 2 seemed hallucinated


### **Scenario 5**
        **temperature**=10,
        top_k=25

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input='''
          What is the protocol for managing sepsis in a critical care unit?
            '''
print(RAG(user_input,lcpp_llm,temperature=10,top_k=25))

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(RAG(user_input,lcpp_llm,temperature=10,top_k=25))

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(RAG(user_input,lcpp_llm,temperature=10,top_k=25))

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(RAG(user_input,lcpp_llm,temperature=10,top_k=25))

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(RAG(user_input,lcpp_llm,temperature=10,top_k=25))

####***Observation***
* Model started hallucinating in many responses



### **Scenario 6**
        **temperature**=10,
        top_k=*40*

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input='''
          What is the protocol for managing sepsis in a critical care unit?
            '''
print(RAG(user_input,lcpp_llm,temperature=10,top_k=40))

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(RAG(user_input,lcpp_llm,temperature=10,top_k=40))

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(RAG(user_input,lcpp_llm,temperature=10,top_k=40))

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(RAG(user_input,lcpp_llm,temperature=10,top_k=40))

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(RAG(user_input,lcpp_llm,temperature=10,top_k=40))

####***Observation***
* Model started hallucinating in many responses



* **With varying temperature RAG showed specific and creative responses**
* **Temperature =0 showed more specific responses**
* **Temperature =16 showed more creative responses**


## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same llm model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:

groundedness_rater_system_message = """

You will be presented a ###Question, ###Context used by the AI system and AI generated ###Answer.

Your task is to judge the extent to which the ###Answer is derived from ###Context.

Rate it 1 - if The ###Answer is not derived from the ###Context at all
Rate it 2 - if The ###Answer is derived from the ###Context only to a limited extent
Rate it 3 - if The ###Answer is derived from ###Context to a good extent
Rate it 4 - if The ###Answer is derived from ###Context mostly
Rate it 5 - if The ###Answer is is derived from ###Context completely

Please note: Make sure you give a single overall rating in the range of 1 to 5 along with an overall explanation.

"""

In [ ]:

relevance_rater_system_message = """

You will be presented with a ###Question, the ###Context used by the AI system to generate a response, and the AI-generated ###Answer.

Your task is to judge the extent to which the ###Answer is relevant to the ###Question, considering whether it directly addresses the key aspects of the ###Question based on the provided ###Context.

Rate the relevance as follows:
- Rate 1 – The ###Answer is not relevant to the ###Question at all.
- Rate 2 – The ###Answer is only slightly relevant to the **###Question**, missing key aspects.
- Rate 3 – The ###Answer is moderately relevant, addressing some parts of the **###Question** but leaving out important details.
- Rate 4 – The ###Answer is mostly relevant, covering key aspects but with minor gaps.
- Rate 5 – The ###Answer is fully relevant, directly answering all important aspects of the **###Question** with appropriate details from the **###Context**.

Note: Provide a single overall rating in the range of 1 to 5, along with a brief explanation of why you assigned that score.
"""

In [ ]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [ ]:
def generate_ground_relevance_response(user_input,answer,llm):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)


    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=128,
            temperature= 0,
            top_p= 0.95,
            repeat_penalty= 1.2,
            top_k= 50,
            stop=['INST'],
            echo=False
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens= 128,
            temperature= 0.4,
            top_p= 0.95,
            repeat_penalty= 1.2,
            top_k= 50,
            stop=['INST'],
            echo=False
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input_1 = 'What is the protocol for managing sepsis in a critical care unit?'
answer2 = RAG(user_input_1, lcpp_llm,0,top_k=25)

In [ ]:
groundedness_report_2, relevance_report_2 = generate_ground_relevance_response(user_input_1,answer2,lcpp_llm)

In [ ]:
print(groundedness_report_2, '\n\n', relevance_report_2)

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input = 'What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?'
answer2 = RAG(user_input, lcpp_llm,0,top_k=25)

In [ ]:
groundedness_report_2, relevance_report_2 = generate_ground_relevance_response(user_input_1,answer2,lcpp_llm)

In [ ]:
print(groundedness_report_2, '\n\n', relevance_report_2)

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input = 'What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?'
answer2 = RAG(user_input, lcpp_llm,0,top_k=25)

In [ ]:
groundedness_report_2, relevance_report_2 = generate_ground_relevance_response(user_input,answer2,lcpp_llm)

In [ ]:
print(groundedness_report_2, '\n\n', relevance_report_2)

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input = 'What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?'
answer2 = RAG(user_input, lcpp_llm,0,top_k=25)

In [ ]:
groundedness_report_2, relevance_report_2 = generate_ground_relevance_response(user_input,answer2,lcpp_llm)

In [ ]:
print(groundedness_report_2, '\n\n', relevance_report_2)

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input = 'What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?'
answer2 = RAG(user_input, lcpp_llm,0,top_k=25)

In [ ]:
groundedness_report_2, relevance_report_2 = generate_ground_relevance_response(user_input,answer2,lcpp_llm)

In [ ]:
print(groundedness_report_2, '\n\n', relevance_report_2)

### Evaluate RAG with temperature=0.4

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input_1 = 'What is the protocol for managing sepsis in a critical care unit?'
answer2 = RAG(user_input_1, lcpp_llm,0.4,top_k=25)

In [ ]:
groundedness_report_2, relevance_report_2 = generate_ground_relevance_response(user_input_1,answer2,lcpp_llm)

In [ ]:
print(groundedness_report_2, '\n\n', relevance_report_2)

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input = 'What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?'
answer2 = RAG(user_input, lcpp_llm,0.4,top_k=25)

In [ ]:
groundedness_report_2, relevance_report_2 = generate_ground_relevance_response(user_input_1,answer2,lcpp_llm)

In [ ]:
print(groundedness_report_2, '\n\n', relevance_report_2)

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input = 'What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?'
answer2 = RAG(user_input, lcpp_llm,0.4,top_k=25)

In [ ]:
groundedness_report_2, relevance_report_2 = generate_ground_relevance_response(user_input,answer2,lcpp_llm)

In [ ]:
print(groundedness_report_2, '\n\n', relevance_report_2)

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input = 'What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?'
answer2 = RAG(user_input, lcpp_llm,0.4,top_k=25)

In [ ]:
groundedness_report_2, relevance_report_2 = generate_ground_relevance_response(user_input,answer2,lcpp_llm)

In [ ]:
print(groundedness_report_2, '\n\n', relevance_report_2)

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input = 'What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?'
answer2 = RAG(user_input, lcpp_llm,0.4,top_k=25)

In [ ]:
groundedness_report_2, relevance_report_2 = generate_ground_relevance_response(user_input,answer2,lcpp_llm)

In [ ]:
print(groundedness_report_2, '\n\n', relevance_report_2)

## RAG Evaluation Table

| Query # | Question Summary                                  | Temperature | Groundedness Rating | Relevance Rating |
|--------:|----------------------------------------------------|------------:|---------------------:|------------------:|
| 1 | Protocol for managing sepsis in a critical care unit | 0   | 5 | 4 |
| 2 | Symptoms of appendicitis, medical vs surgical         | 0   | 4 | 4 |
| 3 | Treatments and causes of sudden patchy hair loss      | 0   | 4 | 4 |
| 4 | Treatments for physical brain injury                  | 0   | 4 | 4 |
| 5 | Precautions and treatment for a leg fracture          | 0   | 4 | 5 |
| 1 | Protocol for managing sepsis in a critical care unit | 0.4 | 5 | 4 |
| 2 | Symptoms of appendicitis, medical vs surgical         | 0.4 | 4 | 4 |
| 3 | Treatments and causes of sudden patchy hair loss      | 0.4 | 4 | 4 |
| 4 | Treatments for physical brain injury                  | 0.4 | 4 | 4 |
| 5 | Precautions and treatment for a leg fracture          | 0.4 | 4 | 4 |


| Temperature | Creativity     | Ratings    | Hallucination   | Suitability                     |
|-------------|----------------|--------------|------------------|----------------------------------|
| 0           | Very Low       | Excellent    | Very Low         | ✔ Best for medical / factual    |
| 0.4         | Low            | Very Good    | Low              | ✔ Best balance                  |


* **The evaluation shows temperature=0 has better groundness, relevance ratings than temperature=0.4**

## Actionable Insights and Business Recommendations
* Company could rely on RAG since the response to each question given is more coherent and contextually connected.

* We recommend to apply prompt engineering along with the question to retrieve and organize better contextual information.
* We experimented and tuned RAG  model in six different scenarios. Due to time and resourse limitation we could not do further experiments. However according to current research done, company can use RAG model with the following parameters
* **Chunking**
  * encoding_name='cl100k_base',
  * chunk_size=512,
  * chunk_overlap= 20
* **LLM model**
  - max_tokens=500,
  - temperature=0 or 0.4,
  - top_p=0.80,
  - repeat_penalty=1.2,
  - top_k=25,
  - stop=['INST'],
  - echo=False

* We recommend to consult with medical practioners for credibiliity and if needed tune the model again with different parameter values.  
* Further research can be done by changing the chunk_overlap size, chunking strategies, max_token, top_k and k values.
* We suggest ***temperature value=0***, as the model will be used for medical applications where retrieved information  is critical and used for patient treatments.


<font size=6 color='blue'>Power Ahead</font>
___